# 06 — Long Document Handling — Practical Examples

**Covers**: fixed chunking, overlapping chunks, map-reduce summarization, QA over long docs, semantic chunking

In [ ]:
import os, re
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
enc = tiktoken.encoding_for_model(MODEL)

# Sample long document (simulate a research article)
LONG_DOC = """
Introduction to Large Language Models

Large language models (LLMs) have revolutionized natural language processing since the introduction of the transformer architecture in 2017. These models, trained on massive text corpora using self-supervised learning, have demonstrated remarkable capabilities across a wide range of tasks including text generation, translation, summarization, question answering, and reasoning.

The development of foundational models such as GPT-3, PaLM, and LLaMA has shown that scaling model parameters and training data leads to emergent capabilities. GPT-3, released by OpenAI in 2020, demonstrated that a single model trained with next-token prediction could perform few-shot learning without any task-specific fine-tuning. This was a significant breakthrough because it suggested that sufficiently large models develop internal representations that generalize across domains.

Architecture and Training

Modern LLMs are based on the transformer architecture, which uses self-attention mechanisms to model relationships between tokens in a sequence. The key insight of the transformer is that attention allows the model to dynamically focus on different parts of the input when generating each output token, regardless of distance in the sequence. This overcomes the limitations of recurrent neural networks, which struggled with long-range dependencies.

Training LLMs requires enormous computational resources. GPT-4 is estimated to have been trained on over 1 trillion tokens using thousands of GPUs for several months. The training process typically involves two stages: pre-training on a large unlabeled text corpus, and fine-tuning with human feedback (RLHF) to align the model's outputs with human preferences and safety requirements.

Context Windows and Memory

One of the fundamental constraints of LLMs is the context window — the maximum number of tokens the model can process in a single forward pass. Early models like GPT-2 had context windows of just 1,024 tokens. Modern models have dramatically expanded this: GPT-4 supports 128,000 tokens, Claude 3.5 supports 200,000 tokens, and Gemini 1.5 Pro supports up to 2 million tokens.

However, larger context windows come with computational costs that scale quadratically with sequence length due to the attention mechanism. Techniques like sparse attention, linear attention, and state-space models (e.g., Mamba) have been developed to address this scaling challenge. Additionally, research has shown that models often struggle to retrieve information from the middle of long contexts, a phenomenon known as the 'lost in the middle' problem.

Applications in Agentic Systems

LLMs form the reasoning core of modern agentic AI systems. In these systems, LLMs are given tools (web search, code execution, file access) and iterate through a loop of observation, thought, and action. Context window management becomes particularly critical in agentic settings because the agent must maintain conversational history, tool results, and intermediate reasoning across many iterations.

Production agentic systems employ various context management strategies: sliding windows that discard old messages, summarization that compresses history, retrieval-augmented generation that embeds history and retrieves relevant chunks, and hierarchical memory systems that maintain different levels of detail across different timescales.

Future Directions

The field continues to evolve rapidly. Multimodal models that process text, images, audio, and video are becoming mainstream. Test-time compute scaling allows models to reason longer on difficult problems. Mixture-of-experts architectures enable larger effective model capacity without proportional inference cost increases. Constitutional AI and other alignment techniques are being developed to ensure that increasingly capable models remain safe and helpful.

The integration of LLMs with external memory systems, retrieval engines, and structured knowledge bases is opening new possibilities for building systems that combine the flexibility of language models with the precision and scalability of traditional software engineering.
"""

print(f"Document length: {len(enc.encode(LONG_DOC)):,} tokens | {len(LONG_DOC.split()):,} words")
print('✅ Setup complete')

## 1. Fixed Token Chunking

In [ ]:
def chunk_by_tokens(text: str, chunk_size: int = 300) -> list[str]:
    """Split text into fixed token-size chunks at sentence boundaries."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current, count = [], [], 0
    for sent in sentences:
        t = len(enc.encode(sent))
        if count + t > chunk_size and current:
            chunks.append(' '.join(current))
            current, count = [sent], t
        else:
            current.append(sent)
            count += t
    if current:
        chunks.append(' '.join(current))
    return chunks

chunks = chunk_by_tokens(LONG_DOC, chunk_size=300)
print(f"Fixed chunks (300 tokens): {len(chunks)} chunks")
for i, c in enumerate(chunks):
    toks = len(enc.encode(c))
    print(f"  Chunk {i+1}: {toks:>4} tokens — {c[:60]}...")

## 2. Overlapping Chunks — Prevent Boundary Loss

In [ ]:
def chunk_with_overlap(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    token_ids = enc.encode(text)
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(token_ids), step):
        end = min(start + chunk_size, len(token_ids))
        chunks.append(enc.decode(token_ids[start:end]))
        if end >= len(token_ids): break
    return chunks

no_overlap = chunk_by_tokens(LONG_DOC, 300)
with_overlap = chunk_with_overlap(LONG_DOC, 300, 50)

print(f"Without overlap: {len(no_overlap)} chunks")
print(f"With 50-token overlap: {len(with_overlap)} chunks")
print()
print("Last 30 tokens of chunk 1 vs first 30 tokens of chunk 2 (with overlap):")
tokens = enc.encode(LONG_DOC)
print(f"  Chunk 1 end:   {enc.decode(tokens[270:300])!r}")
print(f"  Chunk 2 start: {enc.decode(tokens[250:280])!r}")
print("  ↑ Overlap ensures sentences aren't split at meaning-losing boundaries")

## 3. Map-Reduce Summarization

In [ ]:
def map_reduce_summarize(text: str, chunk_size: int = 400) -> str:
    chunks = chunk_with_overlap(text, chunk_size, overlap=40)
    print(f"Processing {len(chunks)} chunks...")
    
    # MAP: summarize each chunk
    summaries = []
    for i, chunk in enumerate(chunks, 1):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': 'Summarize this section in 1-2 sentences. Be concise.'},
                {'role': 'user',   'content': chunk}
            ],
            max_tokens=80, temperature=0.0
        )
        summaries.append(r.choices[0].message.content)
        print(f"  ✅ Chunk {i} summarized ({len(enc.encode(chunk))} tokens → {len(enc.encode(summaries[-1]))} tokens)")
    
    # REDUCE: combine all summaries
    combined = '\n'.join(f'Part {i+1}: {s}' for i, s in enumerate(summaries))
    final = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Combine these section summaries into one coherent summary.'},
            {'role': 'user',   'content': combined}
        ],
        max_tokens=250, temperature=0.0
    )
    return final.choices[0].message.content

final_summary = map_reduce_summarize(LONG_DOC)
original_tokens = len(enc.encode(LONG_DOC))
final_tokens    = len(enc.encode(final_summary))
print(f"\n{'─'*55}")
print(f"Original: {original_tokens:,} tokens → Summary: {final_tokens} tokens ({final_tokens/original_tokens:.1%})")
print(f"\nFinal Summary:\n{final_summary}")

## 4. QA Over Long Document

In [ ]:
def qa_long_doc(doc: str, question: str, chunk_size: int = 400) -> str:
    chunks = chunk_with_overlap(doc, chunk_size, 40)
    print(f"Q: {question}")
    print(f"Processing {len(chunks)} chunks for relevant info...")
    
    extracts = []
    for i, chunk in enumerate(chunks, 1):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': 'Extract info relevant to the question. If none, respond exactly NONE.'},
                {'role': 'user',   'content': f'Question: {question}\n\nText:\n{chunk}'}
            ],
            max_tokens=100, temperature=0.0
        )
        extract = r.choices[0].message.content.strip()
        if extract.upper() != 'NONE':
            extracts.append(f'[Chunk {i}]: {extract}')
    
    print(f"Found relevant info in {len(extracts)}/{len(chunks)} chunks")
    
    if not extracts:
        return 'No relevant information found.'
    
    evidence = '\n\n'.join(extracts)
    final = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Answer the question based on the evidence. Be concise.'},
            {'role': 'user',   'content': f'Question: {question}\n\nEvidence:\n{evidence}'}
        ],
        max_tokens=200, temperature=0.0
    )
    return final.choices[0].message.content

questions = [
    "What is the 'lost in the middle' problem in LLMs?",
    "How large is GPT-4's context window?",
    "What are the two stages of LLM training?",
]

for q in questions:
    answer = qa_long_doc(LONG_DOC, q)
    print(f"\nA: {answer}")
    print('─' * 55)

## 5. Semantic Chunking — Split at Paragraph Boundaries

In [ ]:
def semantic_chunk(text: str, min_tokens: int = 100, max_tokens: int = 400) -> list[str]:
    """Chunk at paragraph boundaries for more natural splits."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks, current, count = [], [], 0
    for para in paragraphs:
        t = len(enc.encode(para))
        if count + t > max_tokens and count >= min_tokens:
            chunks.append('\n\n'.join(current))
            current, count = [para], t
        else:
            current.append(para)
            count += t
    if current:
        chunks.append('\n\n'.join(current))
    return chunks

sem_chunks = semantic_chunk(LONG_DOC, min_tokens=100, max_tokens=350)
print(f"Semantic chunks: {len(sem_chunks)}")
for i, c in enumerate(sem_chunks):
    toks = len(enc.encode(c))
    first_line = c.split('\n')[0][:60]
    print(f"  Chunk {i+1}: {toks:>4} tokens — starts: {first_line!r}")

## 📌 Summary

- **Fixed chunking**: split at sentence boundaries every N tokens
- **Overlapping**: 10-15% overlap prevents losing boundary context
- **Map-Reduce**: MAP each chunk independently → REDUCE into final result
- **QA over long docs**: extract-per-chunk → synthesize — works for any doc size
- **Semantic chunking**: split at paragraphs — more natural than fixed-size
- **Choose chunk size**: ~300-500 for extraction, ~1000-2000 for summarization